In [12]:
from tensorflow import keras
from keras import regularizers
from keras.models import Model
from keras.layers import (
    Input, Dense, Conv2D, GlobalAveragePooling2D, Dropout, Flatten,
    Activation, BatchNormalization, Add, Reshape, DepthwiseConv2D, LeakyReLU
)
from keras.utils import plot_model
from keras import backend as K

def _conv_block(inputs, filters, kernel):

    x = Conv2D(filters, kernel, padding='same', kernel_regularizer=regularizers.l2(0.0001), use_bias=False)(inputs)
    x = BatchNormalization(axis=1)(x)
    x = LeakyReLU()(x)
    return x

def _bottleneck_block(inputs, filters, kernel, factor):
    expanded_filters = filters * factor
    x = _conv_block(inputs, filters=expanded_filters, kernel=(1, 1))

    x = DepthwiseConv2D(kernel, padding='same', kernel_regularizer=regularizers.l2(0.0001), use_bias=False)(x)
    x = BatchNormalization(axis=1)(x)
    x = LeakyReLU()(x)

    x = _conv_block(x, filters=filters, kernel=(1, 1))
    x = Add()([x, inputs])

    return x


def GoMobileNetv2(input_shape, filters, factor, block_num):
    
    inputs = Input(shape=input_shape)
    x = _conv_block(inputs, filters, (1, 1))

    for i in range(0, block_num):
        x = _bottleneck_block(x, filters, (3,3), factor)


    # Policy head
    policy_head = Conv2D(1, 1, activation='relu', padding='same', use_bias=False,
                                kernel_regularizer=regularizers.l2(0.0001))(x)
    policy_head = Flatten()(policy_head)
    policy_head = Activation('softmax', name='policy')(policy_head)

    # Value head
    value_head = GlobalAveragePooling2D()(x)
    value_head = Dense(50, kernel_regularizer=regularizers.l2(0.0001))(value_head)
    value_head = LeakyReLU()(value_head)
    value_head = Dropout(0.3)(value_head)
    value_head = Dense(1, activation='sigmoid', name='value',
                              kernel_regularizer=regularizers.l2(0.0001))(value_head)

    model = keras.Model(inputs=inputs, outputs=[policy_head, value_head])
    return model


model = GoMobileNetv2((19,19,31), 64, 4, 5)
print(model.summary())


Model: "model_9"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_12 (InputLayer)       [(None, 19, 19, 31)]         0         []                            
                                                                                                  
 conv2d_178 (Conv2D)         (None, 19, 19, 64)           1984      ['input_12[0][0]']            
                                                                                                  
 batch_normalization_248 (B  (None, 19, 19, 64)           76        ['conv2d_178[0][0]']          
 atchNormalization)                                                                               
                                                                                                  
 leaky_re_lu_257 (LeakyReLU  (None, 19, 19, 64)           0         ['batch_normalization_24